# Data Scraping: API Calls for Energy Prices, Weather Actuals and Weather Forecasts

This notebook documents the data collection pipeline for the DK1 electricity price forecasting project.
All raw data is fetched from two public APIs:

| Source | API | Data |
|--------|-----|------|
| Energi Data Service | `api.energidataservice.dk` | Day-ahead electricity prices, hourly consumption |
| Open-Meteo | `archive-api.open-meteo.com` | Historical weather actuals (2021–present) |
| Open-Meteo | `previous-runs-api.open-meteo.com` | NWP previous-run forecasts (2025–present) |

The entry point for all collection is `src/data/data_collection.py`. The four fetch functions can be called individually or together via `fetch_all()`.

In [ ]:
import sys
from pathlib import Path

# Make sure the project root is on the path
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

---
## 1. Day-ahead electricity prices

**Source:** Energi Data Service — two overlapping datasets
- `Elspotprices`: historical spot prices up to ~2023
- `DayAheadPrices`: the newer dataset covering more recent observations

Both datasets are fetched, normalised to the same column schema, merged, and deduplicated.  
The result contains one row per hour per price area with prices in both EUR/MWh and DKK/MWh.

**Why DK1?**  
DK1 covers western Denmark. Prices there are strongly influenced by wind power availability,
interconnection flows to Germany and Norway, and varying demand levels — which makes price
forecasting both relevant and challenging.

**Coverage:** 2021-01-01 to present  
**Output:** `data/day_ahead_prices_dk1_raw.csv`

In [ ]:
from src.data.data_collection import fetch_day_ahead_prices

# Fetch (or re-fetch) prices — skips the API call if you just want to read the existing file
# prices_df = fetch_day_ahead_prices(start="2021-01-01", end="2026-04-28", price_area="DK1")

In [ ]:
# Read the already-collected file
prices_df = pd.read_csv("data/day_ahead_prices_dk1_raw.csv", parse_dates=["TimeDK", "TimeUTC"])
print(f"Rows: {len(prices_df):,}")
print(f"Range: {prices_df['TimeDK'].min().date()}  →  {prices_df['TimeDK'].max().date()}")
print(f"Source datasets present: {prices_df['_source_dataset'].unique()}")
prices_df.head()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

daily = (
    prices_df.set_index("TimeDK")["DayAheadPriceDKK"]
    .resample("D").mean()
)

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(daily.index, daily.values, linewidth=0.8, color="#1565C0")
ax.set(ylabel="DKK/MWh", title="DK1 day-ahead electricity price (daily average)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

**Reading the price series:**  
The 2022 energy crisis is visible as an extreme spike. After 2023 prices return to a more
normal range. This motivates the post-crisis filtering used later in the uncertainty estimation step,
where only residuals from 2023 onwards are used to estimate confidence bands.

---
## 2. Hourly electricity consumption

**Source:** Energi Data Service — `ConsumptionIndustry` dataset  
**Endpoint:** `api.energidataservice.dk/dataset/ConsumptionIndustry`

The API returns consumption observations at the grid-area level. The project uses price-area
aggregation to DK1, giving a single consumption figure per hour.

Consumption is used as a proxy for grid stress: hours where system-level demand is high are
treated as peak-load periods. The household flexibility simulation later estimates how much
of that peak is potentially shiftable.

**Coverage:** 2021-01-01 to present  
**Output:** `data/consumption_dk1_raw.csv`

In [ ]:
from src.data.data_collection import fetch_consumption

# Fetch consumption for DK1
# consumption_df = fetch_consumption(start="2021-01-01", end="2026-04-28", price_area="DK1")

In [ ]:
consumption_df = pd.read_csv("data/consumption_dk1_raw.csv", parse_dates=["TimeDK"])
print(f"Rows: {len(consumption_df):,}")
print(f"Range: {consumption_df['TimeDK'].min().date()}  →  {consumption_df['TimeDK'].max().date()}")
print(f"Grid areas: {consumption_df['GridArea'].nunique()}")
consumption_df.head()

In [ ]:
hourly = (
    consumption_df.groupby("TimeDK")["ConsumptionkWh"]
    .sum()
    .resample("D").sum()
    / 1e6  # kWh → GWh
)

fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(hourly.index, hourly.values, linewidth=0.8, color="#2E7D32")
ax.set(ylabel="GWh/day", title="DK1 total electricity consumption (daily sum)")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

---
## 3. Weather actuals

**Source:** Open-Meteo Historical Weather API  
**Endpoint:** `https://archive-api.open-meteo.com/v1/archive`  
**Model:** ECMWF IFS (kept consistent with the forecast model family)

Eight hourly weather variables are fetched for DK1 West (lat 56.15, lon 8.45):

| Variable | Unit | Relevance |
|----------|------|-----------|
| `wind_speed_10m` | m/s | Wind turbine production proxy |
| `wind_direction_10m` | ° | Wind direction (encoded as sin/cos in model) |
| `wind_speed_100m` | m/s | Hub-height wind — closer to turbine operating height |
| `wind_direction_100m` | ° | Hub-height direction |
| `shortwave_radiation` | W/m² | Solar PV production proxy |
| `cloud_cover` | % | Affects solar production |
| `temperature_2m` | °C | Heating / cooling demand |
| `pressure_msl` | hPa | Large-scale weather regime signal |

The actuals serve two roles:
1. **Error distribution estimation** — compared against real NWP forecasts to characterise how large forecast errors are at each horizon
2. **Synthetic forecast generation** — noise drawn from those error distributions is added to create simulated forecasts for the years before the Previous Runs API data is available

**Coverage:** 2021-01-01 to present  
**Output:** `data/weather_actuals_raw.csv`

In [ ]:
from src.data.data_collection import fetch_weather_actuals

# Fetch historical weather for DK1 West
# actuals_df = fetch_weather_actuals(start="2021-01-01", end="2026-04-28")

In [ ]:
actuals_df = pd.read_csv("data/weather_actuals_raw.csv", parse_dates=["TimeDK"])
print(f"Rows: {len(actuals_df):,}")
print(f"Regions: {actuals_df['region'].unique()}")
print(f"Range: {actuals_df['TimeDK'].min().date()}  →  {actuals_df['TimeDK'].max().date()}")
actuals_df.head()

In [ ]:
# Quick overview: one week of selected weather variables
sample = actuals_df.set_index("TimeDK").sort_index().last("7D")

fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
plot_vars = [
    ("wind_speed_100m",     "m/s",  "Wind speed 100 m",      "#1565C0"),
    ("shortwave_radiation", "W/m²", "Shortwave radiation",   "#F57F17"),
    ("temperature_2m",      "°C",   "Temperature 2 m",       "#C62828"),
    ("cloud_cover",         "%",    "Cloud cover",           "#546E7A"),
]

for ax, (var, unit, label, color) in zip(axes, plot_vars):
    ax.plot(sample.index, sample[var], linewidth=1.2, color=color)
    ax.set_ylabel(f"{label}\n({unit})", fontsize=9)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
fig.suptitle("DK1 West — last 7 days of weather actuals", fontsize=12)
fig.tight_layout()
plt.show()

---
## 4. NWP weather forecasts (Previous Runs API)

**Source:** Open-Meteo Previous Runs API  
**Endpoint:** `https://previous-runs-api.open-meteo.com/v1/forecast`  
**Model:** ECMWF IFS

The Previous Runs API stores the *as-issued* NWP forecast for each day, going back up to about
12 months. For each target hour it returns six columns:

- `<var>` — Day 0 forecast (issued on the same day as the target)
- `<var>_previous_day1` — forecast issued 1 day before the target
- `<var>_previous_day2` — forecast issued 2 days before the target
- ... up to `_previous_day5` (120-hour lead time)

These are **real forecast values** — what a forecaster would have seen — not retrospective
reanalysis. This makes them the gold-standard source for estimating how large NWP errors
actually are at each lead time.

**Limitation:** Data is only available from approximately January 2025 onwards.  
This yields ~5 000 matched rows (forecast vs actual) per variable per horizon — enough to
characterise the error distribution, but not for multi-year training.  
For years before 2025, synthetic forecast values are generated in the data processing step.

**Coverage:** 2025-01-01 to present  
**Output:** `data/weather_forecasts_raw.csv`

In [ ]:
from src.data.data_collection import fetch_weather_forecasts

# Fetch NWP previous-run forecasts (only available from 2025-01-01)
# forecasts_df = fetch_weather_forecasts(start="2025-01-01", end="2026-04-28")

In [ ]:
forecasts_df = pd.read_csv("data/weather_forecasts_raw.csv", parse_dates=["TimeDK"])
print(f"Rows: {len(forecasts_df):,}")
print(f"Regions: {forecasts_df['region'].unique()}")
print(f"Range: {forecasts_df['TimeDK'].min().date()}  →  {forecasts_df['TimeDK'].max().date()}")
# Show the previous-day columns for one variable
cols_to_show = ["TimeDK", "region", "temperature_2m",
                "temperature_2m_previous_day1", "temperature_2m_previous_day3", "temperature_2m_previous_day5"]
forecasts_df[[c for c in cols_to_show if c in forecasts_df.columns]].head()

In [ ]:
# Compare actual vs forecast temperature at different horizons for the most recent month
merged = forecasts_df.merge(actuals_df[["TimeDK", "region", "temperature_2m"]],
                            on=["TimeDK", "region"], suffixes=("_fcst", "_actual"))
recent = merged.set_index("TimeDK").sort_index().last("30D")

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

for ax, (day_col, label, color) in zip(axes, [
    ("temperature_2m_previous_day1", "24h forecast",  "#1E88E5"),
    ("temperature_2m_previous_day5", "120h forecast", "#E53935"),
]):
    if day_col not in recent.columns:
        ax.set_title(f"{label} — column not found")
        continue
    ax.plot(recent.index, recent["temperature_2m_actual"], color="black",
            linewidth=1.2, label="Actual")
    ax.plot(recent.index, recent[day_col], color=color, linewidth=1.2,
            linestyle="--", label=label)
    ax.set_ylabel("°C")
    ax.legend(fontsize=8)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
fig.suptitle("DK1 West — actual vs NWP temperature forecast (last 30 days)", fontsize=12)
fig.tight_layout()
plt.show()

**Reading the comparison:**  
The 24-hour forecast (blue dashed) tracks the actual temperature closely.
The 120-hour forecast (red dashed) follows the same broad trends but with larger deviations,
especially during rapid weather transitions. This growing error with horizon is the core
justification for estimating horizon-specific error distributions in the data processing step.

---
## 5. Running the full data collection pipeline

All four sources can be fetched in one call using `fetch_all()`. This is the recommended
way to refresh data from scratch.

```
# From the terminal (recommended for initial setup)
python src/data/data_collection.py --start 2021-01-01 --end 2026-04-28 --area DK1
```

Or from Python / this notebook:

In [ ]:
from src.data.data_collection import fetch_all

# Uncomment to re-fetch everything — this makes live API calls and may take a few minutes
# results = fetch_all(start="2021-01-01", end="2026-04-28", price_area="DK1")
#
# results is a dict with keys:
#   "weather_actuals"   -> data/weather_actuals_raw.csv
#   "weather_forecasts" -> data/weather_forecasts_raw.csv  (from 2025-01-01)
#   "consumption"       -> data/consumption_dk1_raw.csv
#   "prices"            -> data/day_ahead_prices_dk1_raw.csv

---
## 6. Summary of collected files

After running `fetch_all()`, the `data/` directory contains:

In [ ]:
import os

raw_files = [
    "data/day_ahead_prices_dk1_raw.csv",
    "data/consumption_dk1_raw.csv",
    "data/weather_actuals_raw.csv",
    "data/weather_forecasts_raw.csv",
]

rows = []
for fpath in raw_files:
    p = Path(fpath)
    if p.exists():
        df = pd.read_csv(p, nrows=0)  # header only for column count
        size_mb = p.stat().st_size / 1e6
        n_rows  = sum(1 for _ in open(p)) - 1  # fast line count
        rows.append({"File": p.name, "Rows": f"{n_rows:,}",
                     "Columns": len(df.columns), "Size (MB)": f"{size_mb:.1f}"})
    else:
        rows.append({"File": p.name, "Rows": "—", "Columns": "—", "Size (MB)": "—"})

pd.DataFrame(rows)

---
## 7. Next step: Data processing

The raw files feed directly into the data processing pipeline (`src/data/data_processing.py`),
which does two things:

1. **`build_error_distributions()`** — merges the weather forecasts against the actuals,
   computes signed errors at five lead times (24 h, 48 h, 72 h, 96 h, 120 h), and saves
   summary statistics to `data/weather_error_distributions.csv`.

2. **`build_forecast_dataset()`** — for every 12-hour issue time from 2021 to present,
   simulates a 120-hour forecast by adding horizon-scaled Gaussian noise (sampled from
   the error distributions) to the weather actuals. The result is saved as
   `data/forecast_dataset.parquet` and is the direct input to the XGBoost model.

Run the processing step from the terminal:

```
python src/data/data_processing.py
```

Or call the functions individually in a notebook:

```python
from src.data.data_processing import build_error_distributions, build_forecast_dataset

build_error_distributions()
build_forecast_dataset()
```